# Challenge Five: Deploying an Agent to Agent Platform

This notebook reuses the **Cloud Security Advisor** `SequentialAgent` from Challenge Four and
deploys it to **Vertex AI Agent Engine** (Agent Platform), then tests the deployed (remote) agent.

Steps:

1. Rebuild the agent (Greeter -> Search -> Critique -> Refine).
2. Deploy it to Agent Engine with `client.agent_engines.create`.
3. Run a test query against the deployed agent and print its streamed events.

> Runtime target: Agent Platform Colab Enterprise (Vertex AI authenticated).

## Architecture

<img src="https://raw.githubusercontent.com/olorinhill/agent-dev-skills-workshop-david/main/challenge-4/adk_cloud_security_advisor_architecture.svg" alt="Cloud Security Advisor SequentialAgent architecture" width="760">

## Step 1: Install dependencies

Versions are pinned for reproducibility. The `[agent_engines,adk]` extra installs the Agent Engine
deployment client surface used later in this notebook.

In [ ]:
%%writefile requirements.txt
ipykernel==7.3.0
jupyter==1.1.1
pandas==3.0.3
google-adk==1.18.0
litellm==1.83.9
google-cloud-aiplatform[agent_engines,adk]==1.148.1
requests==2.32.3

%pip install -q -r requirements.txt
print("Dependencies installed. If imports fail below, restart the runtime and continue.")

## Step 2: Configuration and authentication

Sets the GCP project, region, and model identifier, and routes all GenAI/ADK calls through Vertex AI. In Colab Enterprise the runtime service account authenticates automatically, so no key file is needed for Gemini.

In [ ]:
import os

import vertexai

# --- GCP / project settings ---
PROJECT_ID = "qwiklabs-gcp-02-b0d878248173"
LOCATION = "us-central1"

# --- Model identifier ---
GEMINI_MODEL = "gemini-2.5-flash"

# --- Route GenAI/ADK through Vertex AI ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Project: {PROJECT_ID}")
print(f"Gemini model: {GEMINI_MODEL} ({LOCATION})")

## Step 3: Define the four sub-agents

Same agent team as Challenge Four. Each `LlmAgent` stores its result in shared session state via
`output_key`, and downstream agents read those values with `{state_key}` instruction templating.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search

greeter_agent = Agent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description="Acknowledges the user's question and hands off to the workflow.",
    instruction=(
        "You are the greeter for the Cloud Security Advisor. In one friendly sentence, "
        "acknowledge the user's question and say that the team will research, critique, and "
        "refine an answer. Do NOT attempt to answer the question yourself."
    ),
    output_key="greeting",
)

search_agent = Agent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Researches and drafts the initial answer using Google Search.",
    instruction=(
        "You are a Google Cloud security researcher. Use google_search to research the user's "
        "question, then write a thorough, well-structured initial answer with concrete, "
        "actionable best practices."
    ),
    tools=[google_search],
    # google_search cannot be combined with an auto-added transfer tool (Gemini 400 error).
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
    output_key="initial_answer",
)

critique_agent = Agent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Reviews the initial answer and recommends improvements.",
    instruction=(
        "You are a senior cloud security reviewer. Review the initial answer below for accuracy, "
        "completeness, clarity, and missing security considerations. Provide specific, actionable "
        "improvement recommendations as a bulleted list. Do not rewrite the answer.\n\n"
        "INITIAL ANSWER:\n{initial_answer}"
    ),
    output_key="critique",
)

refine_agent = Agent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Rewrites the answer based on the critique.",
    instruction=(
        "Rewrite the answer to incorporate the critique. Produce ONLY the final, improved, "
        "well-structured response.\n\n"
        "ORIGINAL ANSWER:\n{initial_answer}\n\n"
        "CRITIQUE:\n{critique}"
    ),
    output_key="final_answer",
)

print("Defined greeter_agent, search_agent, critique_agent, refine_agent.")

## Step 4: Assemble the SequentialAgent

`SequentialAgent` runs its sub-agents in the exact order provided. This `root_agent` is what gets
deployed to Agent Platform.

In [ ]:
from google.adk.agents.sequential_agent import SequentialAgent

cloud_security_advisor = SequentialAgent(
    name="cloud_security_advisor",
    description="Greets, researches, critiques, and refines a security answer in strict order.",
    sub_agents=[greeter_agent, search_agent, critique_agent, refine_agent],
)

root_agent = cloud_security_advisor
print(f"Built SequentialAgent '{root_agent.name}' with stages:")
for index, sub_agent in enumerate(root_agent.sub_agents, start=1):
    print(f"  {index}. {sub_agent.name}")

## Step 5: Vertex client and staging bucket

Agent Engine stages the packaged agent in a Cloud Storage bucket before building it. This cell
creates the bucket if it does not already exist, then initializes the Vertex AI client used to
deploy and manage the agent.

In [ ]:
from google.cloud import storage

BUCKET_NAME = f"{PROJECT_ID}-agent-engine-staging"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"

storage_client = storage.Client(project=PROJECT_ID)
if storage_client.lookup_bucket(BUCKET_NAME) is None:
    storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"Created staging bucket {STAGING_BUCKET}")
else:
    print(f"Using existing staging bucket {STAGING_BUCKET}")

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)
print("Vertex AI client ready.")

## Step 6: Deploy the agent to Agent Platform

Wrap `root_agent` in an `AdkApp` and deploy it with `client.agent_engines.create`. The deployment
packages the agent, stages it to the bucket, builds a container, and provisions a managed runtime.
**This step is asynchronous and can take several minutes.**

In [ ]:
from vertexai import agent_engines

adk_app = agent_engines.AdkApp(agent=root_agent, enable_tracing=True)

remote_agent = client.agent_engines.create(
    agent=adk_app,
    config={
        "display_name": "cloud_security_advisor",
        "description": "Sequential workflow: greet, research, critique, and refine a security answer.",
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-adk==1.18.0",
        ],
        "staging_bucket": STAGING_BUCKET,
    },
)

print("Deployed to Agent Platform.")
print(f"Resource name: {remote_agent.api_resource.name}")

## Step 7: Test the deployed agent

Create a session on the **remote** agent and stream a query against it. Printing the events grouped
by `author` shows each sub-agent (greeter -> search -> critique -> refine) running on Agent Platform,
not locally.

In [ ]:
QUESTION = "What are the best practices for securing a private GKE cluster handling sensitive data?"

remote_session = remote_agent.create_session(user_id="student")
session_id = remote_session["id"] if isinstance(remote_session, dict) else remote_session.id

final_answer = ""
print(f"QUESTION: {QUESTION}\n" + "=" * 80)
for event in remote_agent.stream_query(user_id="student", session_id=session_id, message=QUESTION):
    if not isinstance(event, dict):
        continue
    author = event.get("author", "unknown")
    for part in (event.get("content") or {}).get("parts", []) or []:
        if not isinstance(part, dict):
            continue
        text = part.get("text")
        if text:
            final_answer = text
            print(f"\n----- [{author}] -----\n{text}")

assert final_answer.strip(), "No response produced by the deployed agent"
print("\n" + "=" * 80)
print("Deployed agent responded successfully.")

## Step 8 (optional): Clean up the deployment

Uncomment and run to delete the deployed Agent Engine and avoid lingering resources after grading.

In [ ]:
# remote_agent.delete(force=True)
# print("Deleted the deployed agent.")